# BLIP-2: Score leave-one-query-out ablation (LOQOA)

**Research question:** Which pretrained Q-Former queries contribute most to zero-shot VQAv2 accuracy when the other 31 queries are retained?

This CPU-only notebook analyzes saved predictions from `leave_one_query_out.ipynb`. It uses the same pinned official VQAv2 evaluator as `score_results.ipynb`, without loading BLIP-2, generating predictions, or reading COCO images. The original K=32 reference and all 32 K=31 omissions must belong to the **same completed experiment**.

For query $i\in\{0,\ldots,31\}$, define marginal importance (in percentage points) as

$$I_i=A_{32}-A_{-i}.$$

Positive importance means removing query $i$ lowered accuracy; negative importance means removal improved observed accuracy. This is conditional marginal importance, not an independent measure of query utility or an optimal-pruning ranking.

Run this notebook from the project root or `scripts/` after the inference notebook has finished. The 10-question smoke run verifies the pipeline; the 200-question pilot is exploratory; the 2,000-question main run supports the primary analysis. Keep the query-count scorer and its results separate.

## 1. Select a completed experiment

The default is `smoke`, matching the 10-question default in `leave_one_query_out.ipynb`.

| `STAGE` | Questions per condition | Predictions |
|---|---:|---:|
| `smoke` | 10 | 330 |
| `pilot` | 200 | 6,600 |
| `main` | 2,000 | 66,000 |

Change `STAGE` after completing the corresponding inference stage. `EXPERIMENT_DIR` can be the exact directory printed by the inference notebook (relative to the project root or absolute). Otherwise, the notebook searches completed LOQOA runs for the chosen stage. `GPU_BACKEND` is an optional filter (`"cuda"` or `"rocm"`), referring to the GPU that generated the predictions, not the machine scoring them. If multiple runs match, select the exact directory; no hardware results are pooled.

The bootstrap uses 2,000 replicates by default. Smoke intervals are included to verify the code, **not** to draw scientific conclusions from ten questions.

In [ ]:
from pathlib import Path
import copy
import hashlib
import io
import json
import os
import types
import urllib.request
from contextlib import redirect_stdout

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from filelock import FileLock
from IPython.display import display

STAGE = "smoke"  # "smoke" (10), "pilot" (200), or "main" (2000)
EXPERIMENT_DIR = None  # e.g. "results/leave_one_query_out/smoke-<signature>"
GPU_BACKEND = None  # None, "cuda", or "rocm"
BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 425

STAGE_SIZES = {"smoke": 10, "pilot": 200, "main": 2000}
if STAGE not in STAGE_SIZES or GPU_BACKEND not in {None, "cuda", "rocm"}:
    raise ValueError("Invalid stage or GPU backend.")
if BOOTSTRAP_REPLICATES < 2:
    raise ValueError("At least two bootstrap replicates are required.")

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "scripts" else cwd
RESULTS_ROOT = PROJECT_ROOT / "results" / "leave_one_query_out"


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def recorded_backend(exp):
    return exp.get("gpu_backend") or ("rocm" if exp.get("rocm_version") else
                                      "cuda" if exp.get("cuda_version") else "unknown")


if EXPERIMENT_DIR is None:
    candidates = []
    for path in sorted(RESULTS_ROOT.glob(f"{STAGE}-*/complete.json")):
        candidate = load_json(path.parent / "manifest.json")["experiment"]
        if (candidate.get("experiment_type") == "leave_one_query_out"
                and candidate["stage"] == STAGE
                and (GPU_BACKEND is None or recorded_backend(candidate) == GPU_BACKEND)):
            candidates.append((path.parent, candidate))
    if not candidates:
        raise FileNotFoundError(f"No completed LOQOA {STAGE} run found. Run leave_one_query_out.ipynb first.")
    if len(candidates) > 1:
        options = "\n".join(f"  {path} | {exp['gpu_name']} | {recorded_backend(exp)}" for path, exp in candidates)
        raise ValueError("Multiple runs match; set EXPERIMENT_DIR to one of:\n" + options)
    EXPERIMENT_DIR = candidates[0][0]

EXPERIMENT_DIR = (PROJECT_ROOT / Path(EXPERIMENT_DIR)).resolve()
manifest = load_json(EXPERIMENT_DIR / "manifest.json")
experiment = manifest["experiment"]
complete = load_json(EXPERIMENT_DIR / "complete.json")
expected_signature = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()
if expected_signature != manifest["signature"] or complete["signature"] != expected_signature:
    raise ValueError("Manifest or completion signature does not match.")
if experiment.get("experiment_type") != "leave_one_query_out" or experiment["stage"] != STAGE:
    raise ValueError("Selected experiment is not a LOQOA run for the requested stage.")
backend = recorded_backend(experiment)
if GPU_BACKEND is not None and backend != GPU_BACKEND:
    raise ValueError("Selected experiment does not match GPU_BACKEND.")

question_ids = experiment["question_ids"]
conditions = experiment["conditions"]
if len(question_ids) != STAGE_SIZES[STAGE] or len(set(question_ids)) != len(question_ids):
    raise ValueError("Unexpected or duplicate question IDs.")
expected_names = {"k032", *(f"drop-query-{i:02d}" for i in range(32))}
if len(conditions) != 33 or {c["name"] for c in conditions} != expected_names:
    raise ValueError("Expected exactly the 33 LOQOA conditions.")
for c in conditions:
    removed = None if c["name"] == "k032" else int(c["name"].split("-")[-1])
    expected_indices = [i for i in range(32) if i != removed]
    if (c["removed_query"] != removed or c["k"] != len(expected_indices)
            or c["query_indices"] != expected_indices or c["seed"] is not None):
        raise ValueError(f"Unexpected retained query indices: {c['name']}")
if complete["condition_count"] != 33 or complete["questions_per_condition"] != len(question_ids):
    raise ValueError("Completion counts do not match the manifest.")
print(f"Scoring {STAGE}: {len(question_ids):,} questions × 33 conditions | {experiment['gpu_name']} ({backend})")
print(EXPERIMENT_DIR)

## 2. Load the official VQAv2 evaluator

The evaluator revision and SHA256 digest are identical to `score_results.ipynb`. The first use downloads the source if it is not already cached by the main scorer. Its original scoring rules are preserved; only two Python 2 print statements are adapted for Python 3. Scores use all ten human answers and are expressed on a 0–100 scale. No approximate exact-match metric is used for accuracy.

In [ ]:
EVALUATOR_REVISION = "a013f0043c1e2cdc995922dfe257f7149aa9af06"
EVALUATOR_SHA256 = "f08edfcad5be0112500993e245c706b6cb928eadebe203f89f838e5e0d04bec8"
EVALUATOR_URL = (
    f"https://raw.githubusercontent.com/GT-Vision-Lab/VQA/{EVALUATOR_REVISION}/"
    "PythonEvaluationTools/vqaEvaluation/vqaEval.py"
)
cache = PROJECT_ROOT / "results" / "query_ablation" / "evaluator"
cache.mkdir(parents=True, exist_ok=True)
source_path = cache / f"vqaEval-{EVALUATOR_REVISION}.txt"
with FileLock(str(cache / "download.lock"), timeout=60):
    if source_path.exists():
        source_bytes = source_path.read_bytes()
    else:
        with urllib.request.urlopen(EVALUATOR_URL, timeout=60) as response:
            source_bytes = response.read()
    if hashlib.sha256(source_bytes).hexdigest() != EVALUATOR_SHA256:
        raise ValueError("Official evaluator source failed its SHA256 check.")
    source_path.write_bytes(source_bytes)

source = source_bytes.decode("utf-8")
source = source.replace('print "computing accuracy"', 'print("computing accuracy")')
source = source.replace('print "Done computing accuracy"', 'print("Done computing accuracy")')
official_evaluator = types.ModuleType("official_vqa_evaluator")
exec(compile(source, str(source_path), "exec"), official_evaluator.__dict__)
VQAEval = official_evaluator.VQAEval
print("Official VQA evaluator verified:", EVALUATOR_REVISION)

## 3. Validate annotations and score each condition

The notebook verifies the annotation file hash, exact question order, condition identity, image IDs, ten human answers, and consistency between resumable records and exported predictions. It holds the inference notebook's experiment lock while reading completed files, so a partially written or concurrently modified run cannot be scored. Generation metrics are read from saved records; no GPU is used.

In [ ]:
annotations_path = PROJECT_ROOT / "data" / "validation_dataset" / "v2_mscoco_val2014_annotations_subset.json"
if hashlib.sha256(annotations_path.read_bytes()).hexdigest() != experiment["annotation_file_sha256"]:
    raise ValueError("Annotation file differs from the inference manifest.")
annotations = load_json(annotations_path)["annotations"]
annotation_by_id = {int(a["question_id"]): a for a in annotations}
if len(annotation_by_id) != len(annotations) or not set(question_ids).issubset(annotation_by_id):
    raise ValueError("Missing or duplicate annotations.")
for qid in question_ids:
    if len(annotation_by_id[qid]["answers"]) != 10:
        raise ValueError(f"Question {qid} does not have ten annotator answers.")


def score_predictions(predictions):
    if [p["question_id"] for p in predictions] != question_ids:
        raise ValueError("Predictions must match the exact ordered evaluation set.")
    # The official implementation mutates annotation and result objects.
    ground_truth = types.SimpleNamespace(
        qa={qid: copy.deepcopy(annotation_by_id[qid]) for qid in question_ids},
        getQuesIds=lambda: question_ids,
    )
    answers = types.SimpleNamespace(qa={p["question_id"]: copy.deepcopy(p) for p in predictions})
    evaluator = VQAEval(ground_truth, answers, n=6)
    with redirect_stdout(io.StringIO()):
        evaluator.evaluate(quesIds=question_ids)
    return evaluator


summary_rows = []
scored_rows = []
with FileLock(str(EXPERIMENT_DIR / "experiment.lock"), timeout=0):
    if load_json(EXPERIMENT_DIR / "complete.json") != complete or load_json(EXPERIMENT_DIR / "manifest.json") != manifest:
        raise ValueError("Experiment changed after selection. Rerun the configuration cell.")
    for condition in conditions:
        name = condition["name"]
        directory = EXPERIMENT_DIR / name
        status = load_json(directory / "status.json")
        if status["state"] != "complete" or status["completed"] != len(question_ids):
            raise ValueError(f"Incomplete condition: {name}")
        records = [json.loads(line) for line in (directory / "records.jsonl").read_text(encoding="utf-8").splitlines()]
        if [r["question_id"] for r in records] != question_ids:
            raise ValueError(f"Unexpected, duplicate or missing questions: {name}")
        for row in records:
            if (row["k"] != condition["k"] or row["removed_query"] != condition["removed_query"]
                    or row["query_seed"] is not None):
                raise ValueError(f"Condition mismatch: {name}")
            if row["image_id"] != int(annotation_by_id[row["question_id"]]["image_id"]):
                raise ValueError(f"Image ID mismatch: {name}")
            if not isinstance(row["answer"], str):
                raise ValueError("Saved answers must be strings.")
        predictions = load_json(directory / "predictions.json")
        if predictions != [{"question_id": r["question_id"], "answer": r["answer"]} for r in records]:
            raise ValueError(f"Exported predictions disagree with records: {name}")
        frame = pd.DataFrame(records)
        metric_columns = ["generation_seconds", "question_seconds", "peak_allocated_bytes",
                          "peak_reserved_bytes", "generated_token_count"]
        values = frame[metric_columns].apply(pd.to_numeric, errors="raise")
        if (not np.isfinite(values.to_numpy(dtype=float)).all() or (values < 0).any().any()
                or (values.generation_seconds <= 0).any() or (values.question_seconds <= 0).any()):
            raise ValueError(f"Invalid recorded metrics: {name}")
        frame[metric_columns] = values
        evaluator = score_predictions(predictions)
        scores_pct = np.array([evaluator.evalQA[qid] for qid in question_ids], dtype=float)
        if not np.isfinite(scores_pct).all() or np.any((scores_pct < 0) | (scores_pct > 100)):
            raise ValueError(f"Invalid official scores: {name}")
        if not np.isclose(scores_pct.mean(), evaluator.accuracy["overall"], atol=2e-6, rtol=0):
            raise ValueError(f"Official overall and per-question mean disagree: {name}")
        summary_rows.append({
            "condition": name, "k": condition["k"], "removed_query": condition["removed_query"],
            "questions": len(records), "vqa_accuracy_pct": evaluator.accuracy["overall"],
            "mean_latency_seconds": frame.generation_seconds.mean(),
            "median_latency_seconds": frame.generation_seconds.median(),
            "p95_latency_seconds": frame.generation_seconds.quantile(0.95),
            "throughput_questions_per_second": len(frame) / frame.generation_seconds.sum(),
            "peak_allocated_gib": frame.peak_allocated_bytes.max() / 2**30,
            "peak_reserved_gib": frame.peak_reserved_bytes.max() / 2**30,
            "mean_generated_tokens": frame.generated_token_count.mean(),
        })
        for row, score in zip(records, scores_pct):
            scored_rows.append(dict(row, condition=name, vqa_score_pct=score,
                                    answer_type=annotation_by_id[row["question_id"]]["answer_type"]))

summary = pd.DataFrame(summary_rows).sort_values("removed_query", na_position="first").reset_index(drop=True)
summary["removed_query"] = summary["removed_query"].astype("Int64")
per_question = pd.DataFrame(scored_rows)
per_question["removed_query"] = per_question["removed_query"].astype("Int64")
print(f"Scored {len(per_question):,} predictions using official VQAv2 accuracy.")
display(summary[["condition", "k", "removed_query", "vqa_accuracy_pct"]].round(4))

## 4. Paired query importance and uncertainty

All 33 conditions answer the same questions. For each question $j$, calculate $d_{ij}=s_{32,j}-s_{-i,j}$, where $s$ is the official per-question score in percent. Its question-weighted mean equals $I_i$.

Resample **distinct images** with replacement, retaining all questions associated with each sampled image. Use the same bootstrap draws for all 32 omissions, preserving pairing and cross-query dependence. Report 95% pointwise percentile intervals; they are **not multiplicity-adjusted**. Also report an exploratory approximate simultaneous 95% band based on the maximum absolute centered bootstrap deviation across all 32 estimates, which addresses the family of comparisons but is not an exact finite-sample guarantee. Neither interval captures uncertainty over checkpoints or different pretraining seeds.

The smoke stage can yield wide, degenerate, or unstable intervals. Do not rank queries conclusively or claim significance from ten questions. Even on the main stage, selecting the best queries from this sample and testing a new subset requires independent evaluation.

In [ ]:
drop_names = [f"drop-query-{i:02d}" for i in range(32)]
all_names = ["k032", *drop_names]
scores = per_question.pivot(index="question_id", columns="condition", values="vqa_score_pct").reindex(index=question_ids, columns=all_names)
answers = per_question.pivot(index="question_id", columns="condition", values="answer").reindex(index=question_ids, columns=all_names)
if scores.isna().any().any() or answers.isna().any().any():
    raise ValueError("Paired analysis requires complete predictions and scores.")
baseline = scores["k032"].to_numpy(dtype=float)
drop_scores = scores[drop_names].to_numpy(dtype=float)
deltas = baseline[:, None] - drop_scores
image_ids = np.array([int(annotation_by_id[qid]["image_id"]) for qid in question_ids])


def paired_image_bootstrap_matrix(delta, image_ids, replicates, seed):
    """Resample image clusters jointly across all query omissions."""
    unique_images, codes = np.unique(image_ids, return_inverse=True)
    counts = np.bincount(codes)
    cluster_sums = np.zeros((len(unique_images), delta.shape[1]), dtype=float)
    np.add.at(cluster_sums, codes, delta)
    rng = np.random.default_rng(seed)
    draws = np.empty((replicates, delta.shape[1]), dtype=float)
    for r in range(replicates):
        sampled = rng.integers(0, len(unique_images), size=len(unique_images))
        weights = np.bincount(sampled, minlength=len(unique_images))
        draws[r] = (weights[:, None] * cluster_sums).sum(axis=0) / (weights * counts).sum()
    return draws


importance_values = deltas.mean(axis=0)
boot = paired_image_bootstrap_matrix(deltas, image_ids, BOOTSTRAP_REPLICATES, BOOTSTRAP_SEED)
lower, upper = np.quantile(boot, [0.025, 0.975], axis=0)
# Approximate simultaneous band for the 32 comparisons, using common draws.
max_error = np.quantile(np.max(np.abs(boot - importance_values), axis=1), 0.95)
importance = pd.DataFrame({
    "removed_query": np.arange(32), "condition": drop_names,
    "questions": len(question_ids), "unique_images": len(np.unique(image_ids)),
    "baseline_accuracy_pct": baseline.mean(), "ablation_accuracy_pct": drop_scores.mean(axis=0),
    "importance_pp": importance_values, "ci_lower_pp": lower, "ci_upper_pp": upper,
    "simultaneous_lower_pp": importance_values - max_error,
    "simultaneous_upper_pp": importance_values + max_error,
})
importance["rank"] = importance.importance_pp.rank(method="min", ascending=False).astype(int)
ranked = importance.sort_values(["importance_pp", "removed_query"], ascending=[False, True]).reset_index(drop=True)
if STAGE == "smoke":
    print("Smoke test only: importance estimates and confidence intervals are not reliable research findings.")
display(ranked.round(4))

## 5. Answer-type sensitivity and question-level changes

For each omitted query, calculate accuracy and importance separately for `yes/no`, `number`, and `other`. Report the number of questions per answer type; an absent category is left missing (NaN), not assigned zero accuracy. Subgroup estimates are descriptive and do not establish that a query specializes in a particular type of reasoning.

For question-level changes, compare official VQA scores against K=32: an omission can improve, preserve, or worsen a score. Exact answer-string agreement is a separate measure; different answers may receive the same official VQA score.

In [ ]:
answer_types = ["yes/no", "number", "other"]
question_types = np.array([annotation_by_id[qid]["answer_type"] for qid in question_ids])
if not set(question_types).issubset(answer_types):
    raise ValueError("Unexpected VQAv2 answer type.")
answer_type_rows = []
change_rows = []
base_answers = answers["k032"].to_numpy(dtype=object)
for i, name in enumerate(drop_names):
    delta = deltas[:, i]
    change_rows.append({
        "removed_query": i, "condition": name, "questions": len(question_ids),
        "improved_pct": 100 * np.mean(delta < 0),
        "unchanged_pct": 100 * np.mean(delta == 0),
        "worsened_pct": 100 * np.mean(delta > 0),
        "exact_answer_agreement_pct": 100 * np.mean(answers[name].to_numpy(dtype=object) == base_answers),
    })
    for answer_type in answer_types:
        mask = question_types == answer_type
        count = int(mask.sum())
        answer_type_rows.append({
            "removed_query": i, "condition": name, "answer_type": answer_type, "questions": count,
            "baseline_accuracy_pct": baseline[mask].mean() if count else np.nan,
            "ablation_accuracy_pct": drop_scores[mask, i].mean() if count else np.nan,
            "importance_pp": delta[mask].mean() if count else np.nan,
        })
answer_type_summary = pd.DataFrame(answer_type_rows)
change_summary = pd.DataFrame(change_rows)
assert np.allclose(change_summary[["improved_pct", "unchanged_pct", "worsened_pct"]].sum(axis=1), 100)
display(answer_type_summary.pivot(index="removed_query", columns="answer_type", values="importance_pp").round(3))
display(change_summary.round(3))

## 6. Export numerical results and scoring protocol

All outputs are placed in `analysis/` inside the selected LOQOA experiment directory. No existing query-count analysis is modified. The protocol documents the score sign, bootstrap method, selected manifest, and evaluator revision. Rerunning this notebook regenerates the analysis only.

In [ ]:
report_dir = EXPERIMENT_DIR / "analysis"
report_dir.mkdir(exist_ok=True)
summary.to_csv(report_dir / "run_summary.csv", index=False)
per_question.to_csv(report_dir / "per_question_scores.csv", index=False)
importance.to_csv(report_dir / "query_importance.csv", index=False)
ranked.to_csv(report_dir / "ranked_query_importance.csv", index=False)
answer_type_summary.to_csv(report_dir / "accuracy_by_answer_type.csv", index=False)
change_summary.to_csv(report_dir / "question_level_changes.csv", index=False)
protocol = {
    "experiment_signature": manifest["signature"], "experiment_type": "leave_one_query_out",
    "stage": STAGE, "questions_per_condition": len(question_ids), "condition_count": 33,
    "gpu_name": experiment["gpu_name"], "gpu_backend": backend,
    "evaluator_revision": EVALUATOR_REVISION, "evaluator_sha256": EVALUATOR_SHA256,
    "score_units": "percent", "importance_units": "percentage points",
    "importance_definition": "K=32 baseline accuracy minus accuracy after removing query i",
    "bootstrap_replicates": BOOTSTRAP_REPLICATES, "bootstrap_seed": BOOTSTRAP_SEED,
    "resampling_unit": "distinct image; all associated questions retained; question-weighted mean",
    "pointwise_interval": "95% percentile; not multiplicity-adjusted",
    "simultaneous_interval": "approximate 95% band using maximum absolute centered bootstrap deviation across 32 queries",
    "agreement_definition": "exact saved answer-string equality",
    "answer_type_missing": "NaN if no questions in category",
    "interpretation": "conditional marginal importance, not independent or optimal subset importance",
}
(report_dir / "scoring_protocol.json").write_text(json.dumps(protocol, indent=2) + "\n", encoding="utf-8")
print("Tables saved to:", report_dir)

## 7. Visualize individual query importance

The first figure shows all 32 omissions in their original query-index order, and the second ranks them by observed marginal importance. Positive values indicate an accuracy loss after removal. Both figures display the pointwise bootstrap interval; the ranked figure additionally displays the approximate simultaneous band. A vertical or horizontal line at zero represents no accuracy difference. Smoke/pilot figures are labeled with their sample size.

In [ ]:
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
label = f"{STAGE} | {len(question_ids):,} questions"


def save_show(fig, filename):
    fig.tight_layout()
    fig.savefig(report_dir / filename, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# Draw confidence intervals directly rather than assuming percentile bounds contain the estimate.
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(importance.removed_query, importance.importance_pp)
ax.vlines(importance.removed_query, importance.ci_lower_pp, importance.ci_upper_pp,
          color="black", linewidth=1.1, label="95% pointwise image-bootstrap CI")
ax.scatter(importance.removed_query, importance.importance_pp, color="black", s=9, zorder=3)
ax.axhline(0, color="gray", linewidth=1)
ax.set(xlabel="Removed query index (0–31)", ylabel="Accuracy loss (pp)",
       title=f"Individual query importance — {label}", xticks=np.arange(32))
ax.legend(fontsize=9)
save_show(fig, "query_importance.png")

fig, ax = plt.subplots(figsize=(9, 11))
y = np.arange(len(ranked))
ax.hlines(y, ranked.simultaneous_lower_pp, ranked.simultaneous_upper_pp,
          color="lightgray", linewidth=4, label="Approx. 95% simultaneous band")
ax.hlines(y, ranked.ci_lower_pp, ranked.ci_upper_pp, linewidth=1.8,
          label="95% pointwise CI")
ax.scatter(ranked.importance_pp, y, s=24, zorder=3)
ax.axvline(0, color="gray", linestyle="--")
ax.set(yticks=y, yticklabels=[f"Query {i:02d}" for i in ranked.removed_query],
       xlabel="Accuracy loss relative to K=32 (pp)", title=f"Ranked query importance — {label}")
ax.invert_yaxis()
ax.legend(fontsize=9, loc="best")
save_show(fig, "ranked_query_importance.png")

## 8. Visualize importance by answer type

The diverging heatmap is centered at zero. Each row represents a removed query and each column an answer type. Positive values indicate that removing the query reduced the corresponding category's accuracy. Categories absent from a smoke sample are marked `N/A` and left gray. These are descriptive subgroup comparisons.

In [ ]:
heat = answer_type_summary.pivot(index="removed_query", columns="answer_type", values="importance_pp").reindex(index=range(32), columns=answer_types)
counts = {a: int((question_types == a).sum()) for a in answer_types}
values = heat.to_numpy(dtype=float)
finite = values[np.isfinite(values)]
limit = max(float(np.max(np.abs(finite))) if finite.size else 0.0, 0.01)
cmap = plt.get_cmap("RdBu_r").copy()
cmap.set_bad("lightgray")
fig, ax = plt.subplots(figsize=(7, 12))
image = ax.imshow(np.ma.masked_invalid(values), cmap=cmap, vmin=-limit, vmax=limit, aspect="auto")
ax.set(xticks=np.arange(3), xticklabels=[f"{a} (n={counts[a]})" for a in answer_types],
       yticks=np.arange(32), yticklabels=[f"{i:02d}" for i in range(32)],
       ylabel="Removed query", title=f"Answer-type query importance — {label}")
for i in range(32):
    for j in range(3):
        value = values[i, j]
        ax.text(j, i, f"{value:+.2f}" if np.isfinite(value) else "N/A",
                ha="center", va="center", fontsize=8,
                color="white" if np.isfinite(value) and abs(value) > 0.7 * limit else "black")
fig.colorbar(image, ax=ax, label="Accuracy loss (pp)")
save_show(fig, "query_importance_by_answer_type.png")

## 9. Visualize question-level changes and answer agreement

The stacked bars describe how each omission changes official question scores. The separate agreement plot shows exact generated-answer agreement with the full model. A high agreement rate does not necessarily imply equal VQA accuracy, and vice versa.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = change_summary.removed_query.to_numpy()
bottom = np.zeros(len(x))
for col, title in [("improved_pct", "Improved"), ("unchanged_pct", "Unchanged"), ("worsened_pct", "Worsened")]:
    vals = change_summary[col].to_numpy()
    ax.bar(x, vals, bottom=bottom, label=title)
    bottom += vals
ax.set(xlabel="Removed query", ylabel="Questions (%)", ylim=(0, 100), xticks=np.arange(32),
       title=f"Question-level VQA score changes vs. K=32 — {label}")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
save_show(fig, "question_level_changes.png")

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(x, change_summary.exact_answer_agreement_pct)
ax.set(xlabel="Removed query", ylabel="Exact answer agreement (%)", ylim=(0, 100),
       xticks=np.arange(32), title=f"Answer agreement with K=32 — {label}")
save_show(fig, "prediction_agreement.png")

## 10. Computational efficiency (optional descriptive comparison)

Use generation timing and peak allocated GPU memory recorded during inference. These measurements exclude scoring and do not represent full experiment wall time; that is recorded in `runtime.json` by the inference notebook. The dashed horizontal lines show the same-run K=32 reference. All observations come from the GPU identified in the manifest. Small differences can reflect answer length, hardware scheduling, or measurement noise rather than query importance.

In [ ]:
reference = summary.loc[summary.condition == "k032"].iloc[0]
drops = summary.loc[summary.condition != "k032"].sort_values("removed_query")
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, column, ylabel in [
    (axes[0], "median_latency_seconds", "Median generation latency (s)"),
    (axes[1], "throughput_questions_per_second", "Generation throughput (questions/s)"),
    (axes[2], "peak_allocated_gib", "Peak allocated VRAM (GiB)"),
]:
    ax.plot(drops.removed_query.astype(int), drops[column], marker="o", markersize=3, linewidth=1)
    ax.axhline(reference[column], color="gray", linestyle="--", label="K=32 baseline")
    ax.set(ylabel=ylabel)
    ax.legend(fontsize=9)
axes[-1].set(xlabel="Removed query", xticks=np.arange(32))
fig.suptitle(f"LOQOA generation measurements — {experiment['gpu_name']} | {label}")
save_show(fig, "loqoa_efficiency.png")

## 11. Interpretation and next steps

- A positive importance estimate means the model performed worse when that query was removed, **conditional on all 31 other queries being present**. Negative values are possible, and small differences may be sampling noise.
- The ranked table is exploratory. Marginal effects are not additive; the most influential individual queries do not automatically form the best small subset. If using the ranking to choose queries for a new pruning experiment, evaluate that choice on independent data.
- Pointwise intervals are not simultaneous; consider the approximate simultaneous band when discussing all 32 comparisons. Neither establishes universal query importance.
- On the 10-question smoke stage, verify that all tables and figures exist rather than interpreting the ranking. Move to 200 questions for exploratory patterns and 2,000 questions for the main study if the runtime is acceptable.
- This notebook is independent of `score_results.ipynb`. The original query-count experiment and LOQOA can be scored in either order, and neither requires rerunning model inference when analysis changes.